# Colab 01 - Fase exploratória

Este notebook organiza a fase exploratória do trabalho a partir do notebook original `TPF_INF_493 (1).ipynb`.

Objetivos:

- gerar uma representação 2D por SVD para o conjunto de dados;
- comparar KMeans, DBSCAN e clustering hierárquico;
- salvar tabelas de métricas internas/externas;
- salvar gráficos para apoiar a justificativa da escolha do KMeans.

## 1. Montar Google Drive e configurar caminhos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DATASET_PATH = Path('/content/drive/MyDrive/SBCUP/Dataset/train_test_network.parquet')
RESULTADOS_DIR = Path('/content/drive/MyDrive/SBCUP/Resultados/')
GRAFICOS_DIR = Path('/content/drive/MyDrive/SBCUP/Gráficos/')

EXPLORATORIO_DIR = RESULTADOS_DIR / 'exploratorio'
EXPLORATORIO_DIR.mkdir(parents=True, exist_ok=True)
GRAFICOS_DIR.mkdir(parents=True, exist_ok=True)

assert DATASET_PATH.exists(), f'Dataset não encontrado: {DATASET_PATH}'
print('Dataset:', DATASET_PATH)
print('Resultados exploratórios:', EXPLORATORIO_DIR)
print('Gráficos:', GRAFICOS_DIR)

## 2. Instalar dependências e configurar parâmetros

As amostras seguem a lógica do notebook original: KMeans usa uma amostra maior, DBSCAN e hierárquico usam amostras menores por custo computacional.

In [ ]:
!pip -q install numpy pandas scipy scikit-learn matplotlib seaborn pyarrow

RSEED = 42
KMEANS_SAMPLE = 20000
DBSCAN_SAMPLE = 6000
HIER_SAMPLE = 8000
K_MIN = 2
K_MAX = 60

## 3. Imports e funções auxiliares

In [ ]:
import gc
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse.csgraph import connected_components
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import AgglomerativeClustering, DBSCAN, MiniBatchKMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
    v_measure_score,
)
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid', context='notebook')
np.random.seed(RSEED)


def salvar_fig(nome, dpi=220):
    path = GRAFICOS_DIR / nome
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    plt.show()
    print('Figura salva em:', path)


class PortsConnectivityFeaturizer(BaseEstimator, TransformerMixin):
    def __init__(self, topN=20, log_counts=True, drop_raw_ips=True):
        self.topN = topN
        self.log_counts = log_counts
        self.drop_raw_ips = drop_raw_ips

    def fit(self, X, y=None):
        X = X.copy()
        X['src_port'] = pd.to_numeric(X['src_port'], errors='coerce')
        X['dst_port'] = pd.to_numeric(X['dst_port'], errors='coerce')
        self.top_src_ = set(X['src_port'].value_counts().head(self.topN).index)
        self.top_dst_ = set(X['dst_port'].value_counts().head(self.topN).index)
        X['src_ip'] = X['src_ip'].astype(str).fillna('NA')
        X['dst_ip'] = X['dst_ip'].astype(str).fillna('NA')
        pair = X['src_ip'] + '->' + X['dst_ip']
        self.vc_pair_ = pair.value_counts()
        self.vc_src_ = X['src_ip'].value_counts()
        self.vc_dst_ = X['dst_ip'].value_counts()
        self.src_unique_dst_ = X.groupby('src_ip')['dst_ip'].nunique()
        self.dst_unique_src_ = X.groupby('dst_ip')['src_ip'].nunique()
        return self

    def transform(self, X):
        X = X.copy()
        X['src_port'] = pd.to_numeric(X['src_port'], errors='coerce')
        X['dst_port'] = pd.to_numeric(X['dst_port'], errors='coerce')
        X['src_port_wk'] = (X['src_port'] < 1024).astype('int8')
        X['dst_port_wk'] = (X['dst_port'] < 1024).astype('int8')
        X['src_port_top'] = X['src_port'].where(X['src_port'].isin(self.top_src_), other='other').astype(str)
        X['dst_port_top'] = X['dst_port'].where(X['dst_port'].isin(self.top_dst_), other='other').astype(str)
        X = X.drop(columns=['src_port', 'dst_port'])
        X['src_ip'] = X['src_ip'].astype(str).fillna('NA')
        X['dst_ip'] = X['dst_ip'].astype(str).fillna('NA')
        pair = X['src_ip'] + '->' + X['dst_ip']
        X['pair_count'] = pair.map(self.vc_pair_).fillna(0).astype(float)
        X['src_out_count'] = X['src_ip'].map(self.vc_src_).fillna(0).astype(float)
        X['dst_in_count'] = X['dst_ip'].map(self.vc_dst_).fillna(0).astype(float)
        X['src_unique_dst'] = X['src_ip'].map(self.src_unique_dst_).fillna(0).astype(float)
        X['dst_unique_src'] = X['dst_ip'].map(self.dst_unique_src_).fillna(0).astype(float)
        if self.log_counts:
            for c in ['pair_count', 'src_out_count', 'dst_in_count', 'src_unique_dst', 'dst_unique_src']:
                X[c] = np.log1p(X[c])
        if self.drop_raw_ips:
            X = X.drop(columns=['src_ip', 'dst_ip'], errors='ignore')
        return X


def montar_embedding(df, n_components=50):
    drop = ['label', 'type', 'split', 'cluster_kmeans30', 'cluster_name', 'purity_valid', 'purity_test']
    X_raw = df.drop(columns=drop, errors='ignore').copy()
    feat = PortsConnectivityFeaturizer(topN=20, log_counts=True, drop_raw_ips=True)
    X = feat.fit_transform(X_raw)

    num_cols = X.select_dtypes(include=['number', 'bool']).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    for c in cat_cols:
        X[c] = X[c].astype('string')

    already_logged = {'pair_count', 'src_out_count', 'dst_in_count', 'src_unique_dst', 'dst_unique_src'}
    skew_cols = [c for c in num_cols if c not in ['src_port_wk', 'dst_port_wk'] and c not in already_logged]

    preprocess = ColumnTransformer(
        transformers=[
            ('num', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
                ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
                ('scaler', StandardScaler(with_mean=False)),
            ]), skew_cols),
            ('num_flags', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
                ('scaler', StandardScaler(with_mean=False)),
            ]), [c for c in num_cols if c not in skew_cols]),
            ('cat', Pipeline([
                ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
                ('ohe', OneHotEncoder(handle_unknown='ignore', min_frequency=10)),
            ]), cat_cols),
        ],
        remainder='drop',
    )

    embedder = Pipeline([
        ('preprocess', preprocess),
        ('svd', TruncatedSVD(n_components=n_components, random_state=RSEED)),
        ('scaler', StandardScaler()),
    ])
    Z = embedder.fit_transform(X)
    return Z, embedder


def amostrar_indices(n, size, seed=RSEED):
    size = min(int(size), int(n))
    rng = np.random.RandomState(seed)
    return rng.choice(n, size=size, replace=False)


def internal_metrics(Ev, labels, ignore_noise=True, max_sil_samples=5000, seed=RSEED):
    lab = np.asarray(labels)
    if ignore_noise and np.any(lab == -1):
        mask = lab != -1
        Ev2, lab2 = Ev[mask], lab[mask]
    else:
        Ev2, lab2 = Ev, lab
    uniq = np.unique(lab2)
    if Ev2.shape[0] < 2 or len(uniq) < 2:
        return {'silhouette': np.nan, 'davies_bouldin': np.nan, 'calinski_harabasz': np.nan,
                'n_points_eval': Ev2.shape[0], 'n_clusters_eval': len(uniq)}
    sil_sample = min(max_sil_samples, Ev2.shape[0])
    sil = silhouette_score(Ev2, lab2, sample_size=sil_sample, random_state=seed)
    return {
        'silhouette': float(sil),
        'davies_bouldin': float(davies_bouldin_score(Ev2, lab2)),
        'calinski_harabasz': float(calinski_harabasz_score(Ev2, lab2)),
        'n_points_eval': int(Ev2.shape[0]),
        'n_clusters_eval': int(len(uniq)),
    }


def external_metrics(labels, y_type, y_label, suffix=''):
    labels = np.asarray(labels)
    return {
        f'ARI_type{suffix}': adjusted_rand_score(y_type, labels),
        f'AMI_type{suffix}': adjusted_mutual_info_score(y_type, labels, average_method='arithmetic'),
        f'V_type{suffix}': v_measure_score(y_type, labels),
        f'ARI_label{suffix}': adjusted_rand_score(y_label, labels),
        f'AMI_label{suffix}': adjusted_mutual_info_score(y_label, labels, average_method='arithmetic'),
        f'V_label{suffix}': v_measure_score(y_label, labels),
    }


def cluster_size_stats(labels):
    labels = np.asarray(labels)
    labels = labels[labels != -1]
    if labels.size == 0:
        return {'min_size': 0, 'p1_size': 0.0, 'n_small_lt_10': 0, 'n_small_lt_50': 0, 'max_size': 0}
    vc = pd.Series(labels).value_counts()
    return {
        'min_size': int(vc.min()),
        'p1_size': float(np.percentile(vc.values, 1)),
        'n_small_lt_10': int((vc < 10).sum()),
        'n_small_lt_50': int((vc < 50).sum()),
        'max_size': int(vc.max()),
    }

## 4. Carregar dataset e gerar embedding SVD

In [ ]:
df = pd.read_parquet(DATASET_PATH)
print('df.shape:', df.shape)
display(df[['label', 'type']].head())

Z_all, embedder = montar_embedding(df, n_components=50)
y_type_all = df['type'].astype(str).reset_index(drop=True)
y_label_all = df['label'].astype(str).reset_index(drop=True)

print('Embedding:', Z_all.shape)
try:
    svd = embedder.named_steps['svd']
    print('Variância explicada acumulada em 2 componentes:', svd.explained_variance_ratio_[:2].sum())
    print('Variância explicada acumulada em 50 componentes:', svd.explained_variance_ratio_.sum())
except Exception as e:
    print('Não foi possível reportar variância explicada:', e)

## 5. Representação 2D por SVD para todo o conjunto de dados

In [ ]:
# Usa os dois primeiros componentes do embedding SVD para visualizar todo o conjunto.
plot_df = pd.DataFrame({
    'svd1': Z_all[:, 0],
    'svd2': Z_all[:, 1],
    'type': y_type_all,
})

plt.figure(figsize=(10, 7))
sns.scatterplot(data=plot_df, x='svd1', y='svd2', hue='type', s=5, alpha=0.45, linewidth=0, palette='tab10')
plt.xlabel('SVD componente 1')
plt.ylabel('SVD componente 2')
plt.legend(title='type', bbox_to_anchor=(1.02, 1), loc='upper left')
salvar_fig('svd_2d_todo_dataset_por_type.png')

kmeans30_all = MiniBatchKMeans(n_clusters=30, random_state=RSEED, batch_size=4096, n_init=10)
clusters_k30_all = kmeans30_all.fit_predict(Z_all)
plot_df['cluster_kmeans30'] = clusters_k30_all.astype(str)

plt.figure(figsize=(10, 7))
sns.scatterplot(data=plot_df, x='svd1', y='svd2', hue='cluster_kmeans30', s=5, alpha=0.45, linewidth=0, palette='tab20', legend=False)
plt.xlabel('SVD componente 1')
plt.ylabel('SVD componente 2')
salvar_fig('svd_2d_todo_dataset_por_cluster_kmeans30.png')

del plot_df
gc.collect()

## 6. KMeans: varredura de k

In [ ]:
idx_km = amostrar_indices(len(Z_all), KMEANS_SAMPLE)
E_km = Z_all[idx_km]
y_type_km = y_type_all.iloc[idx_km].reset_index(drop=True)
y_label_km = y_label_all.iloc[idx_km].reset_index(drop=True)

rows = []
for k in range(K_MIN, K_MAX + 1):
    km = MiniBatchKMeans(n_clusters=k, random_state=RSEED, batch_size=4096, n_init=10)
    lab = km.fit_predict(E_km)
    rows.append({
        'algo': 'KMeans', 'param': f'k={k}', 'k': k, 'sample_size': len(E_km), 'noise_ratio': 0.0,
        'inertia': float(km.inertia_),
        **internal_metrics(E_km, lab, ignore_noise=False),
        **external_metrics(lab, y_type_km, y_label_km),
        **cluster_size_stats(lab),
    })

metrics_kmeans = pd.DataFrame(rows)
metrics_kmeans.to_csv(EXPLORATORIO_DIR / 'metricas_kmeans.csv', index=False)
display(metrics_kmeans.sort_values('AMI_type', ascending=False).head(10))

for col, ylabel, fname in [
    ('silhouette', 'Silhouette', 'kmeans_silhouette_por_k.png'),
    ('davies_bouldin', 'Davies-Bouldin', 'kmeans_davies_bouldin_por_k.png'),
    ('AMI_type', 'AMI type', 'kmeans_ami_type_por_k.png'),
    ('V_type', 'V-measure type', 'kmeans_v_type_por_k.png'),
    ('inertia', 'Inércia', 'kmeans_inercia_por_k.png'),
]:
    plt.figure(figsize=(7, 4))
    plt.plot(metrics_kmeans['k'], metrics_kmeans[col], marker='o')
    plt.axvline(30, color='red', linestyle='--', label='k=30')
    plt.xlabel('k')
    plt.ylabel(ylabel)
    plt.legend()
    salvar_fig(fname)

gc.collect()

## 7. DBSCAN: busca exploratória de `eps` e `min_samples`

In [ ]:
idx_db = amostrar_indices(len(Z_all), DBSCAN_SAMPLE)
E_db = StandardScaler().fit_transform(Z_all[idx_db])
y_type_db = y_type_all.iloc[idx_db].reset_index(drop=True)
y_label_db = y_label_all.iloc[idx_db].reset_index(drop=True)


def propose_eps_from_kdist(Ev, min_samples, qs=(0.90, 0.93, 0.95, 0.97, 0.98, 0.99)):
    nn = NearestNeighbors(n_neighbors=min_samples, metric='euclidean', n_jobs=1)
    nn.fit(Ev)
    dists, _ = nn.kneighbors(Ev, return_distance=True)
    dk = np.sort(dists[:, -1])
    return np.unique(np.quantile(dk, qs))

rows = []
for ms in [10, 20, 30]:
    for eps in propose_eps_from_kdist(E_db, ms):
        db = DBSCAN(eps=float(eps), min_samples=int(ms), metric='euclidean', n_jobs=1)
        lab = db.fit_predict(E_db)
        mask = lab != -1
        row = {
            'algo': 'DBSCAN', 'param': f'eps={eps:.4f}, min_samples={ms}',
            'eps': float(eps), 'min_samples': int(ms), 'sample_size': len(E_db),
            'n_clusters': int(len(set(lab)) - (1 if -1 in lab else 0)),
            'noise_ratio': float((lab == -1).mean()),
            **internal_metrics(E_db, lab, ignore_noise=True),
            **external_metrics(lab, y_type_db, y_label_db, suffix='_all'),
            **cluster_size_stats(lab),
        }
        if mask.sum() >= 2 and len(np.unique(lab[mask])) >= 2:
            row.update(external_metrics(lab[mask], y_type_db.iloc[mask], y_label_db.iloc[mask], suffix='_no_noise'))
        rows.append(row)

metrics_dbscan = pd.DataFrame(rows)
metrics_dbscan.to_csv(EXPLORATORIO_DIR / 'metricas_dbscan.csv', index=False)
display(metrics_dbscan.sort_values('AMI_type_all', ascending=False).head(10))

gc.collect()

## 8. Clustering hierárquico/agglomerative

In [ ]:
idx_hi = amostrar_indices(len(Z_all), HIER_SAMPLE)
E_hi = Z_all[idx_hi]
y_type_hi = y_type_all.iloc[idx_hi].reset_index(drop=True)
y_label_hi = y_label_all.iloc[idx_hi].reset_index(drop=True)


def make_connected_knn_graph(Ev, k_start=30, k_step=20, k_max=400):
    last = None
    for k in range(k_start, k_max + 1, k_step):
        conn = kneighbors_graph(Ev, n_neighbors=k, include_self=False, n_jobs=-1)
        n_comp, _ = connected_components(conn)
        last = (conn, int(n_comp), int(k))
        if n_comp == 1:
            return last
    return last

conn, n_comp, k_used = make_connected_knn_graph(E_hi)
print('Componentes no grafo kNN:', n_comp, '| k usado:', k_used)

rows = []
for linkage in ['ward', 'average', 'complete']:
    for k in [9, 10, 20, 30, 40]:
        model = AgglomerativeClustering(n_clusters=int(k), linkage=linkage, connectivity=conn, compute_full_tree=False)
        lab = model.fit_predict(E_hi)
        rows.append({
            'algo': 'Hierárquico', 'param': f'linkage={linkage}, k={k}',
            'linkage': linkage, 'k': int(k), 'sample_size': len(E_hi),
            'n_clusters': int(k), 'noise_ratio': 0.0,
            'knn_components': n_comp, 'k_used_knn': k_used,
            **internal_metrics(E_hi, lab, ignore_noise=False),
            **external_metrics(lab, y_type_hi, y_label_hi),
            **cluster_size_stats(lab),
        })

metrics_hier = pd.DataFrame(rows)
metrics_hier.to_csv(EXPLORATORIO_DIR / 'metricas_hierarquico.csv', index=False)
display(metrics_hier.sort_values('AMI_type', ascending=False).head(10))

del conn
gc.collect()

## 9. Comparativo entre algoritmos

In [ ]:
km_best = metrics_kmeans.loc[metrics_kmeans['AMI_type'].idxmax()].copy()
db_best = metrics_dbscan.loc[metrics_dbscan['AMI_type_all'].idxmax()].copy()
hi_best = metrics_hier.loc[metrics_hier['AMI_type'].idxmax()].copy()

comparativo = pd.DataFrame([km_best, db_best, hi_best]).reset_index(drop=True)
comparativo['AMI_type_comparavel'] = comparativo['AMI_type'].fillna(comparativo.get('AMI_type_all'))
comparativo['V_type_comparavel'] = comparativo['V_type'].fillna(comparativo.get('V_type_all'))
comparativo['AMI_label_comparavel'] = comparativo['AMI_label'].fillna(comparativo.get('AMI_label_all'))
comparativo['V_label_comparavel'] = comparativo['V_label'].fillna(comparativo.get('V_label_all'))
comparativo.to_csv(EXPLORATORIO_DIR / 'comparativo_melhores_configuracoes.csv', index=False)
display(comparativo[['algo', 'param', 'sample_size', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'AMI_type_comparavel', 'V_type_comparavel', 'AMI_label_comparavel', 'V_label_comparavel', 'noise_ratio']])

plot_df = comparativo.copy()
plot_df['label'] = plot_df['algo'].astype(str) + '\n' + plot_df['param'].astype(str)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (col, title) in zip(axes.ravel(), [
    ('silhouette', 'Silhouette (maior melhor)'),
    ('davies_bouldin', 'Davies-Bouldin (menor melhor)'),
    ('AMI_type_comparavel', 'AMI type (maior melhor)'),
    ('V_type_comparavel', 'V-measure type (maior melhor)'),
]):
    ax.bar(plot_df['label'], plot_df[col])
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(GRAFICOS_DIR / 'comparativo_kmeans_dbscan_hierarquico.png', dpi=220, bbox_inches='tight')
plt.show()

## 10. Arquivos gerados

In [ ]:
print('Tabelas em:', EXPLORATORIO_DIR)
for p in sorted(EXPLORATORIO_DIR.glob('*.csv')):
    print('-', p.name)

print('\nGráficos em:', GRAFICOS_DIR)
for p in sorted(GRAFICOS_DIR.glob('*.png')):
    print('-', p.name)